# Gate B — K/τ sweep (Kaggle T4×2)

Default K=32, τ=0.5 already lost to the twin (ΔMSE about −2% at step 650). This session sweeps the remaining kill-rule cells: K ∈ {8, 32, 64}, τ ∈ {0.3, 0.5, 0.7}, skipping 32/0.5. Each arm is 15 minutes from the same Taylor init. Never P100. Never bf16.


In [ ]:
import os
from pathlib import Path

REPO = "https://github.com/Caedral-ai/notrehybrid.git"
cwd = Path.cwd()

if (cwd / "setup_kaggle.sh").exists():
    root = cwd
elif (cwd / "notrehybrid" / "setup_kaggle.sh").exists():
    root = cwd / "notrehybrid"
else:
    !git clone --depth 1 {REPO} notrehybrid
    root = cwd / "notrehybrid"

os.chdir(root)
print("repo root:", root)
!bash setup_kaggle.sh

In [ ]:
WORK = "/kaggle/working"
TEACHER = f"{WORK}/teachers/SmolLM2-360M"
TAYLOR = f"{WORK}/checkpoints/gate-b/init-taylor"
CKPT = f"{WORK}/checkpoints/gate-b"
CFG = "configs/smollm2_360m/gate_a.yaml"
print(TEACHER, TAYLOR, CKPT)

In [ ]:
import os, subprocess, sys
os.environ["PYTHONUNBUFFERED"] = "1"
TEACHER = "/kaggle/working/teachers/SmolLM2-360M"
TAYLOR = "/kaggle/working/checkpoints/gate-b/init-taylor"
CKPT = "/kaggle/working/checkpoints/gate-b"
CFG = "configs/smollm2_360m/gate_a.yaml"
print("CELL3_START", flush=True)

def run(cmd):
    print("RUN", " ".join(cmd), flush=True)
    rc = subprocess.call(cmd)
    print("RC", rc, flush=True)
    if rc != 0:
        raise SystemExit(rc)

run(["python", "-u", "-m", "notre.convert.convert_smollm2", "--hf", "HuggingFaceTB/SmolLM2-360M", "--out", TEACHER])
from pathlib import Path
released = 0
for root in (Path.home() / ".cache" / "huggingface", Path("/root/.cache/huggingface")):
    if not root.exists():
        continue
    for lock in root.rglob("*.lock"):
        lock.unlink(missing_ok=True)
        released += 1
print(f"released {released} huggingface locks", flush=True)
run(["python", "-u", "-m", "notre.convert.taylor_calibrate", "--cfg", CFG, "--output", TAYLOR, "--teacher", TEACHER, "--hf-teacher", "HuggingFaceTB/SmolLM2-360M"])
print("TAYLOR_DONE", flush=True)
for slots in (8, 32, 64):
    for tau in (0.3, 0.5, 0.7):
        if slots == 32 and tau == 0.5:
            print("skip slots=32 tau=0.5 (already measured)", flush=True)
            continue
        tag = f"k{slots}-t{tau}"
        print(f"SWEEP_START {tag}", flush=True)
        run([
            "python", "-u", "-m", "notre.convert.transfer",
            "--cfg", CFG, "--teacher", TEACHER, "--student-init", TAYLOR,
            "--ckpt-dir", f"{CKPT}/sweep/{tag}",
            "--cache", "--slots", str(slots), "--tau", str(tau),
            "--minutes", "15", "--save-every", "100000", "--keep-last", "1",
        ])
        print(f"SWEEP_DONE {tag}", flush=True)
print("SWEEP_ALL_DONE", flush=True)


In [ ]:
print('sweep finished in the previous cell', flush=True)


In [ ]:
from pathlib import Path
root = Path("/kaggle/working/checkpoints/gate-b/sweep")
if not root.exists():
    print("missing", root)
else:
    for path in sorted(root.glob("*/transfer-cache/mse.csv")):
        print("==", path)
        lines = path.read_text().splitlines()
        print("\n".join(lines[:3]))
        print("...")
        print("\n".join(lines[-3:]))
